# Sparse Release Drawdown-Control Checkpoint Score v1

## Purpose

The previous handoff built per-date drawdown-control targets from the failed `wf_2022` PPO path. This notebook asks a narrower question before any retraining: do any existing candidate paths already satisfy those targets and the `+30pp / <=15% MaxDD` gate?

The answer must be checkpoint/candidate scoring evidence, not a promotion shortcut. `2022` remains an evaluation window; this is a diagnostic screen to decide whether the next step should be BC warm start, checkpoint objective, or another training branch.

## External Basis

- CVaR / risk-sensitive sequential control motivates penalizing tail drawdown directly: https://arxiv.org/abs/1506.02188
- Portfolio DRL should treat allocation as a sequential risk/return control problem: https://arxiv.org/abs/1706.10059
- Qlib remains the reproducibility and workflow boundary: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL reinforces explicit environment/training/evaluation separation: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- PBO / CSCV warns against selecting the best-looking candidate without out-of-sample checks: https://doi.org/10.21314/JCF.2016.322

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
import sys

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'AGENTS.md').exists() and (path / 'src').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.soxl_quant.app.state_objective_training_targets import (
    StateObjectiveCheckpointSelectionConfig,
    StateObjectiveTrainingTargetConfig,
    score_state_objective_checkpoint_candidates,
    score_state_objective_target_alignment,
    summarize_state_objective_checkpoint_selection,
    summarize_state_objective_target_alignment,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

EXPERIMENT = 'sparse_release_drawdown_control_checkpoint_score_v1'
OUT_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
TARGET_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / 'sparse_release_drawdown_control_target_handoff_v1'
FAILED_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / 'sparse_release_failed_fold_interval_diagnostic_v1'
GUARD_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / 'sparse_release_drawdown_budget_guard_screen_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 15.0

In [2]:
targets = pd.read_csv(TARGET_DIR / 'drawdown_control_training_targets.csv', parse_dates=['datetime'])
failed_interval_targets = targets[targets['inside_failed_drawdown_interval'].astype(bool)].copy()
path_metrics = pd.read_csv(FAILED_DIR / 'path_metrics.csv')
window_guard_metrics = pd.read_csv(GUARD_DIR / 'window_guard_metrics.csv')
aggregate_guard_metrics = pd.read_csv(GUARD_DIR / 'aggregate_guard_metrics.csv')
reference_returns = pd.read_csv(FAILED_DIR / 'ppo_selected_daily_2022.csv', parse_dates=['datetime'])[['datetime', 'return']]

coverage = {
    'experiment': EXPERIMENT,
    'source_target_experiment': 'sparse_release_drawdown_control_target_handoff_v1',
    'source_failed_fold_experiment': 'sparse_release_failed_fold_interval_diagnostic_v1',
    'source_guard_experiment': 'sparse_release_drawdown_budget_guard_screen_v1',
    'full_year_target_rows': int(len(targets)),
    'failed_interval_target_rows': int(len(failed_interval_targets)),
    'candidate_sources': ['bc_daily_2022', 'ppo_selected_daily_2022', 'drawdown_budget_guard_daily_paths'],
}
coverage

{'experiment': 'sparse_release_drawdown_control_checkpoint_score_v1',
 'source_target_experiment': 'sparse_release_drawdown_control_target_handoff_v1',
 'source_failed_fold_experiment': 'sparse_release_failed_fold_interval_diagnostic_v1',
 'source_guard_experiment': 'sparse_release_drawdown_budget_guard_screen_v1',
 'full_year_target_rows': 250,
 'failed_interval_target_rows': 138,
 'candidate_sources': ['bc_daily_2022',
  'ppo_selected_daily_2022',
  'drawdown_budget_guard_daily_paths']}

In [3]:
def _add_nav_columns(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out['window'] = 'wf_2022'
    if 'strategy_nav' not in out.columns:
        if 'nav' in out.columns:
            out['strategy_nav'] = pd.to_numeric(out['nav'], errors='coerce')
        elif 'equity_nav' in out.columns:
            out['strategy_nav'] = pd.to_numeric(out['equity_nav'], errors='coerce')
    if 'buyhold_nav' not in out.columns:
        returns = pd.to_numeric(out.get('return', 0.0), errors='coerce').fillna(0.0)
        out['buyhold_nav'] = (1.0 + returns).cumprod()
        if len(out) and out['buyhold_nav'].iloc[0] != 0:
            out['buyhold_nav'] = out['buyhold_nav'] / out['buyhold_nav'].iloc[0]
    return out


def _full_path(name: str) -> pd.DataFrame:
    frame = pd.read_csv(FAILED_DIR / name, parse_dates=['datetime'])
    return _add_nav_columns(frame)


def _guard_path(name: str) -> pd.DataFrame:
    frame = pd.read_csv(GUARD_DIR / name, parse_dates=['datetime'])
    frame = frame.merge(reference_returns, on='datetime', how='left')
    return _add_nav_columns(frame)

candidate_daily_paths = {
    'bc_initialization': {'wf_2022': _full_path('bc_daily_2022.csv')},
    'ppo_selected': {'wf_2022': _full_path('ppo_selected_daily_2022.csv')},
}
for path in sorted(GUARD_DIR.glob('wf_2022_*_daily.csv')):
    label = path.name.removeprefix('wf_2022_').removesuffix('_daily.csv')
    candidate_daily_paths[f'guard_{label}'] = {'wf_2022': _guard_path(path.name)}

list(candidate_daily_paths)

['bc_initialization',
 'ppo_selected',
 'guard_cvar_budget_08_15_floor05',
 'guard_cvar_budget_10_15_cash',
 'guard_cvar_budget_10_20_floor10',
 'guard_cvar_budget_12_25_floor15',
 'guard_no_guard']

In [4]:
training_config = StateObjectiveTrainingTargetConfig(
    target_relative_return=TARGET_RELATIVE_RETURN_PP,
    target_max_dd=TARGET_MAX_DD_PCT,
    downtrend_position_cap=0.25,
    max_dd_gap_weight=10.0,
)
checkpoint_config = StateObjectiveCheckpointSelectionConfig(
    alignment_loss_weight=1.0,
    window_gate_loss_weight=0.01,
    missing_target_penalty=10.0,
    rebound_mae_weight=1.0,
    risk_cap_violation_weight=2.0,
    legacy_score_reward=0.0,
)

alignment_full = score_state_objective_target_alignment(candidate_daily_paths, targets, config=training_config)
checkpoint_full = score_state_objective_checkpoint_candidates(alignment_full, config=checkpoint_config)
summary_alignment_full = summarize_state_objective_target_alignment(alignment_full)
summary_checkpoint_full = summarize_state_objective_checkpoint_selection(checkpoint_full)

alignment_interval = score_state_objective_target_alignment(candidate_daily_paths, failed_interval_targets, config=training_config)
checkpoint_interval = score_state_objective_checkpoint_candidates(alignment_interval, config=checkpoint_config)
summary_alignment_interval = summarize_state_objective_target_alignment(alignment_interval)
summary_checkpoint_interval = summarize_state_objective_checkpoint_selection(checkpoint_interval)

alignment_full.to_csv(OUT_DIR / 'alignment_scores_full_year.csv', index=False)
checkpoint_full.to_csv(OUT_DIR / 'checkpoint_scores_full_year.csv', index=False)
alignment_interval.to_csv(OUT_DIR / 'alignment_scores_failed_interval.csv', index=False)
checkpoint_interval.to_csv(OUT_DIR / 'checkpoint_scores_failed_interval.csv', index=False)

checkpoint_full.head(10)

,candidate,target_rows,matched_rows,missing_target_ratio,coverage_gate,position_tracking_mae,rebound_position_mae,risk_cap_violation,window_gate_loss,max_relative_gap,max_drawdown_gap,window_gate,alignment_loss,score,checkpoint_objective_loss,checkpoint_selection_score
0,bc_initialization,250,250,0.000,True,0.209725,0.0,0.016536,42.455366,0.0,3.396429,False,0.700422,0.0,1.158047,-1.158047
1,ppo_selected,250,250,0.000,True,0.000000,0.0,0.157808,296.033669,0.0,23.682694,False,3.591569,0.0,6.867521,-6.867521
2,guard_cvar_budget_10_15_cash,250,191,0.236,False,0.402183,0.0,0.000519,5.723636,0.0,0.457891,False,2.821494,0.0,5.239768,-5.239768
3,guard_cvar_budget_08_15_floor05,250,191,0.236,False,0.352774,0.0,0.000432,57.218646,0.0,4.577492,False,3.286689,0.0,6.219740,-6.219740
4,guard_cvar_budget_10_20_floor10,250,191,0.236,False,0.291006,0.0,0.005272,161.874710,0.0,12.949977,False,4.290842,0.0,8.280133,-8.280133
5,guard_cvar_budget_12_25_floor15,250,191,0.236,False,0.196899,0.0,0.031811,245.822646,0.0,19.665812,False,5.142367,0.0,10.024215,-10.024215
6,guard_no_guard,250,191,0.236,False,0.000000,0.0,0.157808,394.044017,0.0,31.523521,False,6.931672,0.0,13.547728,-13.547728


In [5]:
checkpoint_interval.head(10)

,candidate,target_rows,matched_rows,missing_target_ratio,coverage_gate,position_tracking_mae,rebound_position_mae,risk_cap_violation,window_gate_loss,max_relative_gap,max_drawdown_gap,window_gate,alignment_loss,score,checkpoint_objective_loss,checkpoint_selection_score
0,guard_cvar_budget_10_15_cash,138,138,0.0,True,0.404712,0.0,0.000840,5.723636,0.0,0.457891,False,0.465308,0.0,0.524224,-0.524224
1,bc_initialization,138,138,0.0,True,0.196370,0.0,0.026769,42.455366,0.0,3.396429,False,0.727999,0.0,1.206091,-1.206091
2,guard_cvar_budget_08_15_floor05,138,138,0.0,True,0.355639,0.0,0.000700,57.218646,0.0,4.577492,False,0.930624,0.0,1.504210,-1.504210
3,guard_cvar_budget_10_20_floor10,138,138,0.0,True,0.286712,0.0,0.008535,161.874710,0.0,12.949977,False,1.939599,0.0,3.575415,-3.575415
4,guard_cvar_budget_12_25_floor15,138,138,0.0,True,0.165838,0.0,0.051496,245.822646,0.0,19.665812,False,2.830050,0.0,5.391269,-5.391269
5,ppo_selected,138,138,0.0,True,0.000000,0.0,0.163681,296.033669,0.0,23.682694,False,3.615062,0.0,6.902762,-6.902762
6,guard_no_guard,138,138,0.0,True,0.000000,0.0,0.163681,394.044017,0.0,31.523521,False,4.595166,0.0,8.862969,-8.862969


In [6]:
base_gate_context = path_metrics.rename(columns={'path': 'candidate'}).copy()
base_gate_context['candidate'] = base_gate_context['candidate'].map({
    'bc_initialization': 'bc_initialization',
    'ppo_selected': 'ppo_selected',
}).fillna(base_gate_context['candidate'])
base_gate_context['source'] = 'failed_fold_path_metrics'
base_gate_context['full_gate'] = base_gate_context['gate'].astype(bool)
base_gate_context['max_drawdown_gap'] = (pd.to_numeric(base_gate_context['max_dd'], errors='coerce') - TARGET_MAX_DD_PCT).clip(lower=0.0)
base_gate_context['max_relative_gap'] = (TARGET_RELATIVE_RETURN_PP - pd.to_numeric(base_gate_context['relative_return'], errors='coerce')).clip(lower=0.0)
base_gate_context = base_gate_context[[
    'candidate', 'source', 'full_gate', 'relative_return', 'max_dd', 'avg_pos', 'max_relative_gap', 'max_drawdown_gap'
]]

guard_gate_context = aggregate_guard_metrics.copy()
guard_gate_context['candidate'] = 'guard_' + guard_gate_context['config_name'].astype(str)
guard_gate_context['source'] = 'drawdown_budget_guard_aggregate'
guard_gate_context = guard_gate_context.rename(columns={
    'max_relative_gap': 'max_relative_gap',
    'max_drawdown_gap': 'max_drawdown_gap',
    'mean_avg_pos': 'avg_pos',
})
guard_gate_context['relative_return'] = None
guard_gate_context['max_dd'] = None
guard_gate_context = guard_gate_context[[
    'candidate', 'source', 'full_gate', 'relative_return', 'max_dd', 'avg_pos', 'max_relative_gap', 'max_drawdown_gap', 'worst_window'
]]

candidate_gate_context = pd.concat([base_gate_context, guard_gate_context], ignore_index=True, sort=False)
candidate_gate_context.to_csv(OUT_DIR / 'candidate_gate_context.csv', index=False)
window_guard_metrics.to_csv(OUT_DIR / 'source_window_guard_metrics.csv', index=False)

candidate_gate_context.sort_values(['full_gate', 'max_drawdown_gap', 'max_relative_gap'], ascending=[False, True, True])

/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_7731/532577417.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  candidate_gate_context = pd.concat([base_gate_context, guard_gate_context], ignore_index=True, sort=False)


,candidate,source,full_gate,relative_return,max_dd,avg_pos,max_relative_gap,max_drawdown_gap,worst_window
2,guard_cvar_budget_10_15_cash,drawdown_budget_guard_aggregate,False,NaN,NaN,0.051165,4.007458,0.786990,wf_2023_2024
0,bc_initialization,failed_fold_path_metrics,False,36.419124,18.396429,0.280667,0.000000,3.396429,NaN
3,guard_cvar_budget_08_15_floor05,drawdown_budget_guard_aggregate,False,NaN,NaN,0.085927,5.024431,4.577492,wf_2022
4,guard_cvar_budget_10_20_floor10,drawdown_budget_guard_aggregate,False,NaN,NaN,0.139866,1.752829,12.949977,wf_2022
5,guard_cvar_budget_12_25_floor15,drawdown_budget_guard_aggregate,False,NaN,NaN,0.184300,1.752829,19.665812,wf_2022
1,ppo_selected,failed_fold_path_metrics,False,41.770333,38.682694,0.318036,0.000000,23.682694,NaN
6,guard_no_guard,drawdown_budget_guard_aggregate,False,NaN,NaN,0.259053,1.752829,31.523521,wf_2022


In [7]:
selected_full = checkpoint_full.iloc[0].to_dict()
selected_interval = checkpoint_interval.iloc[0].to_dict()

context_by_candidate = candidate_gate_context.set_index('candidate').to_dict(orient='index')
full_context = context_by_candidate.get(str(selected_full['candidate']), {})
interval_context = context_by_candidate.get(str(selected_interval['candidate']), {})

hard_blockers = []
if not bool(selected_full.get('window_gate', False)):
    hard_blockers.append('full_year_selected_candidate_misses_window_gate')
if not bool(selected_interval.get('window_gate', False)):
    hard_blockers.append('failed_interval_selected_candidate_misses_window_gate')
if not bool(full_context.get('full_gate', False)):
    hard_blockers.append('full_year_selected_candidate_misses_external_gate_context')
if not bool(interval_context.get('full_gate', False)):
    hard_blockers.append('failed_interval_selected_candidate_misses_cross_window_guard_context')
if float(interval_context.get('avg_pos', 1.0) or 1.0) < 0.10:
    hard_blockers.append('failed_interval_selected_guard_is_near_cash')

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'checkpoint_selection'],
    'profile': 'sparse_release_bc_ppo_screen_v1',
    'source_target_experiment': 'sparse_release_drawdown_control_target_handoff_v1',
    'decision': 'checkpoint_objective_no_promotion',
    'promotion_allowed': False,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'selected_full_year_candidate': str(selected_full['candidate']),
    'selected_full_year_checkpoint_objective_loss': float(selected_full['checkpoint_objective_loss']),
    'selected_failed_interval_candidate': str(selected_interval['candidate']),
    'selected_failed_interval_checkpoint_objective_loss': float(selected_interval['checkpoint_objective_loss']),
    'hard_blockers': hard_blockers,
    'next_mechanism': 'run_fold_local_drawdown_control_bc_or_checkpoint_objective_training',
    'external_sources': [
        'https://arxiv.org/abs/1506.02188',
        'https://arxiv.org/abs/1706.10059',
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
    'sidecars': [
        'coverage.json',
        'alignment_scores_full_year.csv',
        'checkpoint_scores_full_year.csv',
        'alignment_scores_failed_interval.csv',
        'checkpoint_scores_failed_interval.csv',
        'candidate_gate_context.csv',
        'source_window_guard_metrics.csv',
        'selection.json',
        'summary.json',
    ],
}

conclusion = [
    f"Full-year target scoring selects `{selection['selected_full_year_candidate']}` with checkpoint loss {selection['selected_full_year_checkpoint_objective_loss']:.3f}, but it still misses the window gate.",
    f"Failed-interval target scoring selects `{selection['selected_failed_interval_candidate']}` with checkpoint loss {selection['selected_failed_interval_checkpoint_objective_loss']:.3f}, but this is the near-cash guard path and still fails cross-window promotion context.",
    "No existing candidate clears both target alignment and +30pp / <=15% MaxDD gates; this remains a training/checkpoint-objective handoff, not a runtime promotion.",
    "Next step should train or checkpoint-score fold-local drawdown-control targets, then re-run walk-forward validation across wf_2022 and wf_2023_2024 before any PPO handoff.",
]

summary = {
    'experiment': EXPERIMENT,
    'decision': selection['decision'],
    'promotion_allowed': False,
    'coverage': coverage,
    'alignment_full_year_summary': summary_alignment_full,
    'checkpoint_full_year_summary': summary_checkpoint_full,
    'alignment_failed_interval_summary': summary_alignment_interval,
    'checkpoint_failed_interval_summary': summary_checkpoint_interval,
    'selected_full_year': selected_full,
    'selected_failed_interval': selected_interval,
    'selected_full_year_gate_context': full_context,
    'selected_failed_interval_gate_context': interval_context,
    'hard_blockers': hard_blockers,
    'selection': selection,
    'conclusion': conclusion,
}

(OUT_DIR / 'coverage.json').write_text(json.dumps(coverage, indent=2, ensure_ascii=False) + chr(10))
(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2, ensure_ascii=False) + chr(10))
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False) + chr(10))

summary

{'experiment': 'sparse_release_drawdown_control_checkpoint_score_v1',
 'decision': 'checkpoint_objective_no_promotion',
 'promotion_allowed': False,
 'coverage': {'experiment': 'sparse_release_drawdown_control_checkpoint_score_v1',
  'source_target_experiment': 'sparse_release_drawdown_control_target_handoff_v1',
  'source_failed_fold_experiment': 'sparse_release_failed_fold_interval_diagnostic_v1',
  'source_guard_experiment': 'sparse_release_drawdown_budget_guard_screen_v1',
  'full_year_target_rows': 250,
  'failed_interval_target_rows': 138,
  'candidate_sources': ['bc_daily_2022',
   'ppo_selected_daily_2022',
   'drawdown_budget_guard_daily_paths']},
 'alignment_full_year_summary': {'decision': 'target_alignment_no_promotion',
  'selected_candidate': 'bc_initialization',
  'selected_alignment_loss': 0.7004221969260744,
  'selected_window_gate': False,
  'selected_coverage_gate': True,
  'selected_missing_target_ratio': 0.0,
  'next_mechanism': 'train_or_checkpoint_score_state_obj

## Notebook Conclusion

This notebook is the durable single-run conclusion for checkpoint/candidate scoring. It deliberately separates:

- full-year target alignment, which prefers the BC path but still misses the drawdown gate;
- failed-interval target alignment, which prefers the near-cash guard path but fails cross-window promotion context;
- promotion evidence, which remains blocked because no candidate clears both target alignment and the `+30pp / <=15% MaxDD` gate.